# 10. POS Tagging & Named Entity Recognition (NER)

## Part 1: Part of Speech (POS) Tagging

### What is POS Tagging?
POS Tagging is the process of labeling each word in a text as corresponding to a particular part of speech (e.g., Noun, Verb, Adjective, Adverb).

### Why is it important?
* **Word Sense Disambiguation:** It helps the machine understand the difference between the same word used in different contexts.
    * *Example:* "I can **park** the car" (Verb) vs. "I walked in the **park**" (Noun).
* **Lemmatization:** To correctly reduce a word to its root, the model needs to know if it's a verb or a noun (e.g., "better" $\rightarrow$ "good" if adj).

### Common Tags (Universal Dependencies)
* **PROPN:** Proper Noun (e.g., "Google", "Shailya")
* **VERB:** Verb (e.g., "run", "eat")
* **NOUN:** Noun (e.g., "laptop", "code")
* **ADJ:** Adjective (e.g., "big", "fast")

---

## Part 2: Named Entity Recognition (NER)

### What is NER?
NER is a sub-task of information extraction that locates and classifies named entities mentioned in unstructured text into pre-defined categories.

### Common Entity Types
* **PERSON:** People, including fictional.
* **ORG:** Companies, agencies, institutions.
* **GPE:** Countries, cities, states.
* **DATE:** Absolute or relative dates or periods.
* **MONEY:** Monetary values, including unit.

### Applications
* **Resume Parsing:** Extracting names, colleges, and companies from CVs.
* **Content Classification:** Tagging news articles by the companies or locations mentioned.

In [1]:
import spacy
from spacy import displacy

In [2]:
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    print('Downloading Model...')
    from spacy.cli import download
    download('en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

In [3]:
# Sample Text (having ambiguity and entities)
text = 'Apple is looking at buying U.K. startup for $1 billion. Shailya visited Google in California yesterday.'
doc = nlp(text)

In [5]:
# POS Tagging:
print(f"{'TOKEN':<12} {'POS':<10} {'EXPLANATION'}")
print('-' * 40)

for token in doc:
    print(f'{token.text:<12} {token.pos_:<10} {spacy.explain(token.pos_)}')

TOKEN        POS        EXPLANATION
----------------------------------------
Apple        PROPN      proper noun
is           AUX        auxiliary
looking      VERB       verb
at           ADP        adposition
buying       VERB       verb
U.K.         PROPN      proper noun
startup      VERB       verb
for          ADP        adposition
$            SYM        symbol
1            NUM        numeral
billion      NUM        numeral
.            PUNCT      punctuation
Shailya      PROPN      proper noun
visited      VERB       verb
Google       PROPN      proper noun
in           ADP        adposition
California   PROPN      proper noun
yesterday    NOUN       noun
.            PUNCT      punctuation


In [6]:
# NER:
print("\n" + "="*40 + "\n")
print(f"{'ENTITY':<20} {'LABEL':<10} {'EXPLANATION'}")
print("-" * 50)

for ent in doc.ents:
    print(f'{ent.text:<20} {ent.label_:<10} {spacy.explain(ent.label_)}')



ENTITY               LABEL      EXPLANATION
--------------------------------------------------
Apple                ORG        Companies, agencies, institutions, etc.
U.K.                 GPE        Countries, cities, states
$1 billion           MONEY      Monetary values, including unit
Shailya              ORG        Companies, agencies, institutions, etc.
Google               ORG        Companies, agencies, institutions, etc.
California           GPE        Countries, cities, states
yesterday            DATE       Absolute or relative dates or periods


In [8]:
# Visualization:
print('\n--- Visualizing Entities ---')
displacy.render(doc, style= 'ent', jupyter= True)


--- Visualizing Entities ---


## Part 3: Customizing NER (Adding Your Own Entities)

Standard models are trained on general web data (Wikipedia, News). They often don't know specific jargon, new tech terms, or personal project names.

### The Solution: `EntityRuler`
Instead of retraining the deep learning model (which takes hours and massive data), we can add a **Rule-based** component to the pipeline.
1. We define a pattern (e.g., "Look for the text 'Learning-Logs'").
2. We assign it a label (e.g., "PROJECT").
3. We tell spaCy to apply this rule *before* or *after* its standard NER.

In [12]:
import spacy
from spacy.pipeline import EntityRuler
from spacy import displacy

# Loading The Model:
nlp = spacy.load('en_core_web_sm')

# Text:
# The Problem: The model doesn't know custom terms.
# "Learning-Logs" is my specific repo, but spaCy might ignore it or guess wrong.
text = 'Shailya is pushing code to the Learning-Logs repository on GitHub.'
doc_before = nlp(text)

print('--- Before Customization ---')
for ent in doc_before.ents:
    print(f'{ent.text} -> {ent.label_}')

--- Before Customization ---
Shailya -> ORG
the Learning-Logs -> ORG
GitHub -> PRODUCT


In [13]:
# The Fix: Adding the EntityRuler
# We create a ruler and add it to the pipeline BEFORE the standard 'ner' component
ruler = nlp.add_pipe('entity_ruler', before= 'ner')

# Defining Patterns:
patterns = [
    {"label": "PROJECT", "pattern": "Learning-Logs"}, # Pattern: Exact match for "Learning-Logs", Label: "PROJECT"
    {"label": "USER", "pattern": [{"LOWER": "shailya"}]} # Pattern: Case-insensitive match for "shailya", Label: "USER"
]

ruler.add_patterns(patterns)

doc_after = nlp(text)

print('\n--- After Customization ---')
for ent in doc_after.ents:
    print(f'{ent.text} -> {ent.label_}')


--- After Customization ---
Shailya -> USER
Learning-Logs -> PROJECT
GitHub -> PRODUCT


In [14]:
# Visualizing:
displacy.render(doc_after, style= 'ent', jupyter= True)